In [2]:
import geopandas as gpd
import pickle as pkl
from typing import cast

path = "../denmark_coastline"
with open(path, "rb") as f:
    landmasses = cast(gpd.GeoDataFrame, pkl.load(f))

In [ ]:
import pygeohash as pgh
from shapely.geometry import box, Polygon
from shapely.strtree import STRtree
from collections import deque

target_precision = 6
BASE32 = "0123456789bcdefghjkmnpqrstuvwxyz"

tree = STRtree(landmasses.geometry)

def box_hash(hash: str):
    min_lat, min_lon, max_lat, max_lon = pgh.get_bounding_box(hash)
    return box(minx=min_lon, miny=min_lat, maxx=max_lon, maxy=max_lat)

def get_children(hash: str):
    return [hash + ch for ch in BASE32]

hashes = deque(BASE32)
cells_on_land: list[Polygon] = []

while hashes:
    hash = hashes.popleft()
    cell = box_hash(hash)

    hit = len(tree.query(cell, predicate="intersects")) > 0
    if not hit:
        continue

    if len(hash) == target_precision:
        cells_on_land.append(cell)
        continue
    
    fully_contained = len(tree.query(cell, predicate="within")) > 0
    if fully_contained:
        children = deque(get_children(hash))
        while children:
            child = children.popleft()

            if len(child) == target_precision:
                cells_on_land.append(box_hash(child))
                continue
            
            children.extend(get_children(child))
        continue

    hashes.extend(get_children(hash))

display(len(cells_on_land))

4022

In [7]:
gdf = gpd.GeoDataFrame(geometry=cells_on_land, crs=4326)
gdf.explore()